In [1]:
import sys
import os
import numpy as np
sys.path.append("../")
import utils.kitti_loader as kitti_loader

In [ ]:
import torch
torch.tensor([1, 2, 3]) == torch.tensor(torch.tensor([1,2,3]))                                                                                                                                                                  

C:\Users\praga\AppData\Local\Temp\ipykernel_13772\1749559852.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor([1, 2, 3]) == torch.tensor(torch.tensor([1,2,3]))


tensor([True, True, True])

In [3]:
import utils.kitti_loader as kitti_loader


In [4]:
lidar = kitti_loader.get_LiDAR(count=1)

In [5]:
from scipy.spatial import cKDTree

In [6]:
lidar =  kitti_loader.lidar_fps(lidar[0])

In [7]:

tree = cKDTree(lidar)

In [8]:
lidar[0].shape

(4,)

In [11]:
res_idx = tree.query_ball_point(lidar[2], r=10)

In [ ]:
# Minimal, correct RPN that proposes MULTIPLE objects
# (matches Faster R-CNN logic, anchor-based, multi-object by design)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import nms

class RPN(nn.Module):
    def __init__(self, in_channels, num_anchors=9):
        super().__init__()

        # shared conv
        self.conv = nn.Conv2d(in_channels, 512, 3, padding=1)

        # A×2 logits: bg / object
        self.cls_logits = nn.Conv2d(512, num_anchors * 2, 1)

        # A×4 bbox deltas
        self.bbox_pred = nn.Conv2d(512, num_anchors * 4, 1)

    def forward(self, features):
        x = F.relu(self.conv(features))
        logits = self.cls_logits(x)      # (B, 2A, H, W)
        deltas = self.bbox_pred(x)       # (B, 4A, H, W)
        return logits, deltas


# ---------- helpers (anchors + decode) ----------

def generate_anchors(scales, ratios):
    anchors = []
    for s in scales:
        for r in ratios:
            w = s * (r ** 0.5)
            h = s / (r ** 0.5)
            anchors.append([-w/2, -h/2, w/2, h/2])
    return torch.tensor(anchors)

def decode_boxes(anchors, deltas):
    xa = (anchors[:, 0] + anchors[:, 2]) / 2
    ya = (anchors[:, 1] + anchors[:, 3]) / 2
    wa = anchors[:, 2] - anchors[:, 0]
    ha = anchors[:, 3] - anchors[:, 1]

    dx, dy, dw, dh = deltas.T

    x = xa + dx * wa
    y = ya + dy * ha
    w = wa * torch.exp(dw)
    h = ha * torch.exp(dh)

    return torch.stack([x - w/2, y - h/2, x + w/2, y + h/2], dim=1)


# ---------- demo: MULTI-object proposals ----------

if __name__ == "__main__":
    B, C, H, W = 1, 256, 32, 32
    A = 9

    backbone_features = torch.randn(B, C, H, W)

    rpn = RPN(C, A)
    cls_logits, bbox_deltas = rpn(backbone_features)

    # reshape
    cls_logits = cls_logits.permute(0, 2, 3, 1).reshape(-1, 2)
    bbox_deltas = bbox_deltas.permute(0, 2, 3, 1).reshape(-1, 4)

    # objectness probability
    scores = F.softmax(cls_logits, dim=1)[:, 1]

    # anchors
    base_anchors = generate_anchors(
        scales=[32, 64, 128],
        ratios=[0.5, 1.0, 2.0]
    )
    anchors = base_anchors.repeat(H * W, 1)

    # decode boxes
    boxes = decode_boxes(anchors, bbox_deltas)

    # NMS → MULTIPLE OBJECTS
    keep = nms(boxes, scores, iou_threshold=0.7)

    final_boxes = boxes[keep]
    final_scores = scores[keep]

    print("Number of detected object proposals:", final_boxes.shape[0])


Number of detected object proposals: 1043


: 

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(boxes[:, :3].detach().numpy())

In [9]:
lidar

array([[ -9.653,  -5.275,   0.119,   0.18 ],
       [ 54.188,  53.797,   1.369,   0.   ],
       [-71.036,  33.466,  -0.895,   0.   ],
       ...,
       [ -8.977,  -4.236,  -0.53 ,   0.14 ],
       [-13.289,   5.671,  -1.847,   0.   ],
       [ 11.539,  -4.421,  -0.562,   0.75 ]], dtype=float32)

In [ ]:
# lidar_points = lidar[res_idx]

: 

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')
fig = plt.subplot(111, projection='3d')
fig.scatter(lidar_points[:,0], lidar_points[:,1], lidar_points[:,2], s=1)

In [ ]:
import open3d as o3d
lidar_points = lidar_points[:, :3]
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lidar_points)
o3d.visualization.draw_geometries([pcd]) 

ImportError: DLL load failed while importing pybind: The specified module could not be found.

In [ ]:
points = lidar

In [ ]:
edge_list = []
edge_features = []

for i in range(len(points)):
    print(points.shape)
    indices = tree.query_ball_point(points[i], 4)
    
    # Remove self-loop
    indices = [idx for idx in indices if idx != i]
    print(f"Point {i} has {len(indices)} neighbors.")
  
    for j in indices:
        edge_list.append([i, j])
        # Store relative coordinates as edge features
        edge_features.append(points[j] - points[i])

edge_index = np.array(edge_list).T  # (2, E)
edge_attr = np.array(edge_features)  # (E, 3)

(8192, 4)
Point 0 has 543 neighbors.
(8192, 4)
Point 1 has 3 neighbors.
(8192, 4)
Point 2 has 15 neighbors.
(8192, 4)
Point 3 has 2 neighbors.
(8192, 4)
Point 4 has 13 neighbors.
(8192, 4)
Point 5 has 3 neighbors.
(8192, 4)
Point 6 has 6 neighbors.
(8192, 4)
Point 7 has 1 neighbors.
(8192, 4)
Point 8 has 43 neighbors.
(8192, 4)
Point 9 has 43 neighbors.
(8192, 4)
Point 10 has 1 neighbors.
(8192, 4)
Point 11 has 3 neighbors.
(8192, 4)
Point 12 has 14 neighbors.
(8192, 4)
Point 13 has 25 neighbors.
(8192, 4)
Point 14 has 32 neighbors.
(8192, 4)
Point 15 has 213 neighbors.
(8192, 4)
Point 16 has 13 neighbors.
(8192, 4)
Point 17 has 63 neighbors.
(8192, 4)
Point 18 has 3 neighbors.
(8192, 4)
Point 19 has 8 neighbors.
(8192, 4)
Point 20 has 76 neighbors.
(8192, 4)
Point 21 has 11 neighbors.
(8192, 4)
Point 22 has 285 neighbors.
(8192, 4)
Point 23 has 7 neighbors.
(8192, 4)
Point 24 has 3 neighbors.
(8192, 4)
Point 25 has 8 neighbors.
(8192, 4)
Point 26 has 6 neighbors.
(8192, 4)
Point 27 ha

In [ ]:
# First, check what you have
print(f"Points shape: {points.shape}")
print(f"Points dtype: {points.dtype}")
print(f"Points type: {type(points)}")
print(f"First point: {points[0]}")

# Make sure it's a numpy array with correct shape
points = np.asarray(points, dtype=np.float64)

# If points is (N, 4) or has extra columns, take only XYZ
if points.shape[1] > 3:
    points = points[:, :3]

# Now visualize
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

line_set = o3d.geometry.LineSet()
line_set.points = o3d.utility.Vector3dVector(points)
line_set.lines = o3d.utility.Vector2iVector(edge_index.T.astype(np.int32))

o3d.visualization.draw_geometries([pcd, line_set])

Points shape: (8192, 4)
Points dtype: float32
Points type: <class 'numpy.ndarray'>
First point: [ 4.111 -4.277 -1.251  0.37 ]


In [ ]:
import sys
import os
import numpy as np
sys.path.append("../")
import utils.kitti_loader as kitti_loader

lidar = kitti_loader.get_LiDAR()

In [ ]:
lidar_one = lidar[0]

In [ ]:
lidar_one.shape

(115384, 4)

In [ ]:
import utils.kitti_loader as kitti_loader
lidar_x = kitti_loader.LiDAR_downsamplex(lidar_one)
lidar_y = kitti_loader.LiDAR_downsampley(lidar_one)
lidar_z = kitti_loader.LiDAR_downsamplez(lidar_one)

AttributeError: module 'utils.kitti_loader' has no attribute 'LiDAR_downsamplex'

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import utils.kitti_loader as kitti_loader
def color_with_inferno(points):
    """Apply inferno colormap based on distance."""
    xyz = points[:, :3]
    dist = np.linalg.norm(xyz, axis=1)
    norm = (dist - dist.min()) / (dist.max() - dist.min())
    colors = plt.cm.inferno(norm)[:, :3]
    return colors

# Create Open3D point clouds
pcd_x = o3d.geometry.PointCloud()
pcd_y = o3d.geometry.PointCloud()
pcd_z = o3d.geometry.PointCloud()

pcd_x.points = o3d.utility.Vector3dVector(lidar_x[:, :3])
pcd_y.points = o3d.utility.Vector3dVector(lidar_y[:, :3])
pcd_z.points = o3d.utility.Vector3dVector(lidar_z[:, :3])

pcd_x.colors = o3d.utility.Vector3dVector(color_with_inferno(lidar_x))
pcd_y.colors = o3d.utility.Vector3dVector(color_with_inferno(lidar_y))
pcd_z.colors = o3d.utility.Vector3dVector(color_with_inferno(lidar_z))

# Translate for side-by-side visualization
shift = 50
pcd_y.translate((shift, 0, 0))
pcd_z.translate((2 * shift, 0, 0))

# Visualize
o3d.visualization.draw_geometries([pcd_x, pcd_y, pcd_z],
    window_name="LiDAR Downsampling (X | Y | Z) - Inferno",
    width=1920, height=720,
    left=50, top=50,
    point_show_normal=False)

In [ ]:
import open3d as o3d

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lidar_z[:, :3])
o3d.visualization.draw_geometries([pcd])

In [ ]:
import utils.kitti_loader as kitti_loader
lidar_z = kitti_loader.LiDAR_downsample(lidar_one)

AttributeError: module 'utils.kitti_loader' has no attribute 'LiDAR_downsample'

In [ ]:
import torch
import torch.nn as nn

class PointGNNIteration(nn.Module):
    """
    Single iteration of Point-GNN following equations:
    Δx_i^t = MLP_h^t(s_i^t)
    e_ij^t = MLP_f^t([x_j - x_i + Δx_i^t, s_j^t])
    s_i^(t+1) = MLP_g^t(Max({e_ij | (i,j) ∈ E})) + s_i^t
    """
    def __init__(self, state_dim=300):
        super().__init__()
        
        # MLP_h: Predict offset Δx_i^t
        self.MLP_h = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)  # Output: [Δx, Δy, Δz]
        )
        
        # MLP_f: Compute edge features e_ij^t
        self.MLP_f = nn.Sequential(
            nn.Linear(3 + state_dim, 300),  # Input: [relative_coords(3), neighbor_state(300)]
            nn.ReLU(),
            nn.Linear(300, 300)
        )
        
        # MLP_g: Update vertex state s_i^(t+1)
        self.MLP_g = nn.Sequential(
            nn.Linear(300, 300),
            nn.ReLU(),
            nn.Linear(300, state_dim)
        )
    
    def forward(self, vertex_states, vertex_coords, edge_index, edge_attr):
        """
        Args:
            vertex_states: (M, 300) - s_i^t for all vertices
            vertex_coords: (M, 3) - x_i for all vertices
            edge_index: (2, E) - graph edges [source_nodes, target_nodes]
            edge_attr: (E, 3) - x_j - x_i (relative coordinates)
        
        Returns:
            new_states: (M, 300) - s_i^(t+1)
        """
        M = vertex_states.size(0)
        
        # ==========================================
        # Equation 1: Δx_i^t = MLP_h^t(s_i^t)
        # ==========================================
        delta_x = self.MLP_h(vertex_states)  # (M, 3)
        
        # ==========================================
        # Equation 2: e_ij^t = MLP_f^t([x_j - x_i + Δx_i^t, s_j^t])
        # ==========================================
        
        # Get source and target nodes
        source_nodes = edge_index[0]  # i indices (E,)
        target_nodes = edge_index[1]  # j indices (E,)
        
        # Adjust relative coordinates with offset
        # edge_attr is (x_j - x_i), add Δx_i^t
        adjusted_coords = edge_attr + delta_x[source_nodes]  # (E, 3)
        
        # Get neighbor states s_j^t
        neighbor_states = vertex_states[target_nodes]  # (E, 300)
        
        # Concatenate: [x_j - x_i + Δx_i^t, s_j^t]
        edge_input = torch.cat([adjusted_coords, neighbor_states], dim=1)  # (E, 303)
        
        # Compute edge features
        edge_features = self.MLP_f(edge_input)  # (E, 300)
        
        # ==========================================
        # Equation 3: s_i^(t+1) = MLP_g^t(Max({e_ij | (i,j) ∈ E})) + s_i^t
        # ==========================================
        
        # Max aggregation: for each vertex i, take max of its edge features
        aggregated = torch.zeros(M, 300, device=vertex_states.device)
        
        for i in range(M):
            # Find all edges where i is the source
            mask = source_nodes == i
            if mask.any():
                # Max pooling over edge features
                aggregated[i] = edge_features[mask].max(dim=0)[0]
        
        # Update state with residual connection
        new_states = self.MLP_g(aggregated) + vertex_states  # (M, 300)
        
        return new_states


# ============================================================
# Complete Point-GNN with T iterations
# ============================================================

class PointGNN(nn.Module):
    """
    Full Point-GNN with T iterations
    """
    def __init__(self, state_dim=300, num_iterations=3):
        super().__init__()
        self.num_iterations = num_iterations
        
        # Create T separate GNN layers (not shared weights)
        self.gnn_layers = nn.ModuleList([
            PointGNNIteration(state_dim) 
            for _ in range(num_iterations)
        ])
        
        # Classification head
        self.MLP_cls = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 4)  # [background, car, pedestrian, cyclist]
        )
        
        # Localization head
        self.MLP_loc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)  # [x, y, z, l, h, w, θ]
        )
    
    def forward(self, vertex_features, vertex_coords, edge_index, edge_attr):
        """
        Args:
            vertex_features: (M, 300) - initial s_i^0
            vertex_coords: (M, 3) - vertex positions x_i
            edge_index: (2, E) - graph connectivity
            edge_attr: (E, 3) - x_j - x_i
        
        Returns:
            classifications: (M, 4)
            bboxes: (M, 7)
        """
        # Initialize state
        states = vertex_features  # s_i^0
        
        # T iterations of GNN
        for t in range(self.num_iterations):
            states = self.gnn_layers[t](
                states,          # s_i^t
                vertex_coords,   # x_i
                edge_index,      # edges
                edge_attr        # x_j - x_i
            )
            # Now states = s_i^(t+1)
        
        # Final predictions
        classifications = self.MLP_cls(states)  # (M, 4)
        bboxes = self.MLP_loc(states)          # (M, 7)
        
        return classifications, bboxes


# ============================================================
# Usage Example
# ============================================================

# Your data from previous steps
vertex_features = torch.randn(1000, 300)  # (M, 300) initial features s_i^0
vertex_coords = torch.randn(1000, 3)      # (M, 3) positions x_i
edge_index = torch.randint(0, 1000, (2, 5000))  # (2, E) edges
edge_attr = torch.randn(5000, 3)          # (E, 3) x_j - x_i

# Create model
model = PointGNN(state_dim=300, num_iterations=3)

# Forward pass
classifications, bboxes = model(
    vertex_features, 
    vertex_coords, 
    edge_index, 
    edge_attr
)

print(f"Classifications: {classifications.shape}")  # (1000, 4)
print(f"Bounding boxes: {bboxes.shape}")           # (1000, 7)

In [ ]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

print(count_up_to(3).next)


AttributeError: 'generator' object has no attribute 'next'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing


class PointGNNLayer(MessagePassing):
    """
    Point-GNN layer with auto-registration mechanism.
    Equation 5 from paper:
    Δx^t_i = MLP^t_h(s^t_i)
    e^t_ij = MLP^t_f([x_j - x_i + Δx^t_i, s^t_j])
    s^t+1_i = MLP^t_g(Max({e^t_ij | (i,j) ∈ E})) + s^t_i
    """
    def __init__(self, state_dim, mlp_f_dims, mlp_g_dims):
        super().__init__(aggr='max')
        
        # MLP_h: predicts auto-registration offset (Δx)
        self.mlp_h = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)  # 3D offset
        )
        
        # MLP_f: computes edge features from [relative_coords + offset, neighbor_state]
        self.mlp_f = nn.Sequential(
            nn.Linear(3 + state_dim, mlp_f_dims[0]),
            nn.ReLU(),
            nn.Linear(mlp_f_dims[0], mlp_f_dims[1])
        )
        
        # MLP_g: updates vertex state from aggregated edge features
        self.mlp_g = nn.Sequential(
            nn.Linear(mlp_g_dims[0], mlp_g_dims[1]),
            nn.ReLU(),
            nn.Linear(mlp_g_dims[1], state_dim)
        )

    def forward(self, s_t, coords, edge_index):
        """
        Args:
            s_t: vertex states (N, state_dim)
            coords: vertex coordinates (N, 3)
            edge_index: graph connectivity (2, E)
        Returns:
            s_t+1: updated vertex states (N, state_dim)
        """
        # Compute auto-registration offset
        delta_x = self.mlp_h(s_t)  # (N, 3)
        
        # Message passing
        return self.propagate(edge_index, s=s_t, coords=coords, delta_x=delta_x)

    def message(self, s_j, coords_i, coords_j, delta_x_i):
        """
        Compute edge features: e^t_ij = MLP_f([x_j - x_i + Δx^t_i, s^t_j])
        """
        # Adjusted relative coordinates with auto-registration
        relative_coords = coords_j - coords_i + delta_x_i  # (E, 3)
        
        # Concatenate with neighbor state
        edge_input = torch.cat([relative_coords, s_j], dim=1)
        
        # Compute edge feature
        return self.mlp_f(edge_input)

    def update(self, aggr_out, s):
        """
        Update: s^t+1_i = MLP_g(Max({e^t_ij})) + s^t_i (residual connection)
        """
        return self.mlp_g(aggr_out) + s


class PointGNN(nn.Module):
    """
    Complete Point-GNN network with T iterations
    """
    def __init__(self, num_classes, state_dim=300, T=3):
        super().__init__()
        self.T = T
        self.num_classes = num_classes
        
        # Create T GNN layers (each iteration has different MLPs)
        self.gnn_layers = nn.ModuleList([
            PointGNNLayer(
                state_dim=state_dim,
                mlp_f_dims=[300, 300],  # For Car: (300, 300)
                mlp_g_dims=[300, 300]
            ) for _ in range(T)
        ])
        
        # Classification head
        self.mlp_cls = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
        # Localization head (per class)
        # Each class predicts 7-DoF bounding box
        self.mlp_loc = nn.ModuleDict({
            f'class_{i}': nn.Sequential(
                nn.Linear(state_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 64),
                nn.ReLU(),
                nn.Linear(64, 7)  # (δx, δy, δz, δl, δh, δw, δθ)
            ) for i in range(num_classes)
        })
    
    def forward(self, s_0, coords, edge_index):
        """
        Args:
            s_0: initial vertex states (N, state_dim)
            coords: vertex coordinates (N, 3)
            edge_index: graph connectivity (2, E)
        Returns:
            class_probs: (N, num_classes)
            bbox_preds: dict of {class_idx: (N, 7)}
        """
        s_t = s_0
        
        # T iterations of GNN
        for t in range(self.T):
            s_t = self.gnn_layers[t](s_t, coords, edge_index)
        
        # Classification
        class_logits = self.mlp_cls(s_t)
        class_probs = F.softmax(class_logits, dim=-1)
        
        # Localization (per-class predictions)
        bbox_preds = {}
        for i in range(self.num_classes):
            bbox_preds[i] = self.mlp_loc[f'class_{i}'](s_t)
        
        return class_probs, bbox_preds


class PointGNNLoss(nn.Module):
    """
    Loss function from Equation 6, 8, 9 in the paper
    """
    def __init__(self, alpha=0.1, beta=10.0, gamma=5e-7):
        super().__init__()
        self.alpha = alpha  # classification loss weight
        self.beta = beta    # localization loss weight
        self.gamma = gamma  # L1 regularization weight
    
    def encode_bbox(self, bbox, vertex_coords, scale_factors):
        """
        Encode bounding box relative to vertex (Equation 7)
        Args:
            bbox: (N, 7) - (x, y, z, l, h, w, θ)
            vertex_coords: (N, 3) - (x_v, y_v, z_v)
            scale_factors: dict with keys (l_m, h_m, w_m, θ_0, θ_m)
        Returns:
            encoded_bbox: (N, 7) - (δx, δy, δz, δl, δh, δw, δθ)
        """
        x, y, z, l, h, w, theta = bbox.unbind(dim=-1)
        x_v, y_v, z_v = vertex_coords.unbind(dim=-1)
        
        l_m = scale_factors['l_m']
        h_m = scale_factors['h_m']
        w_m = scale_factors['w_m']
        theta_0 = scale_factors['theta_0']
        theta_m = scale_factors['theta_m']
        
        delta_x = (x - x_v) / l_m
        delta_y = (y - y_v) / h_m
        delta_z = (z - z_v) / w_m
        delta_l = torch.log(l / l_m)
        delta_h = torch.log(h / h_m)
        delta_w = torch.log(w / w_m)
        delta_theta = (theta - theta_0) / theta_m
        
        return torch.stack([delta_x, delta_y, delta_z, delta_l, 
                           delta_h, delta_w, delta_theta], dim=-1)
    
    def classification_loss(self, pred_probs, gt_labels):
        """
        Cross-entropy loss (Equation 6)
        Args:
            pred_probs: (N, M) - predicted class probabilities
            gt_labels: (N,) - ground truth class indices
        Returns:
            loss: scalar
        """
        # Convert to one-hot
        M = pred_probs.shape[1]
        y_onehot = F.one_hot(gt_labels, num_classes=M).float()
        
        # Cross-entropy: -sum(y * log(p))
        loss = -torch.sum(y_onehot * torch.log(pred_probs + 1e-10), dim=-1)
        return loss.mean()
    
    def localization_loss(self, pred_bbox, gt_bbox, is_object_mask):
        """
        Huber loss for localization (Equation 8)
        Args:
            pred_bbox: (N, 7) - predicted encoded bbox
            gt_bbox: (N, 7) - ground truth encoded bbox
            is_object_mask: (N,) - 1 if vertex is in object, 0 otherwise
        Returns:
            loss: scalar
        """
        # Compute Huber loss element-wise
        huber_loss = F.smooth_l1_loss(pred_bbox, gt_bbox, reduction='none')
        
        # Only count loss for vertices within objects
        huber_loss = huber_loss * is_object_mask.unsqueeze(-1)
        
        # Average over all vertices
        return huber_loss.sum() / (is_object_mask.sum() + 1e-10)
    
    def forward(self, pred_probs, pred_bboxes, gt_labels, gt_bboxes, 
                vertex_coords, is_object_mask, model_params, scale_factors):
        """
        Total loss (Equation 9): l_total = α*l_cls + β*l_loc + γ*l_reg
        
        Args:
            pred_probs: (N, M) - predicted class probabilities
            pred_bboxes: dict of {class_idx: (N, 7)} - predicted bboxes
            gt_labels: (N,) - ground truth class indices
            gt_bboxes: (N, 7) - ground truth bounding boxes
            vertex_coords: (N, 3) - vertex coordinates
            is_object_mask: (N,) - mask for vertices in objects
            model_params: iterator of model parameters for L1 reg
            scale_factors: dict for bbox encoding
        Returns:
            total_loss, loss_dict
        """
        # Classification loss
        l_cls = self.classification_loss(pred_probs, gt_labels)
        
        # Localization loss (only for correct class predictions)
        l_loc = 0
        for class_idx, pred_bbox in pred_bboxes.items():
            # Only compute loss for vertices of this class
            class_mask = (gt_labels == class_idx) & is_object_mask
            
            if class_mask.sum() > 0:
                # Encode ground truth bbox
                gt_bbox_encoded = self.encode_bbox(
                    gt_bboxes[class_mask], 
                    vertex_coords[class_mask],
                    scale_factors
                )
                
                # Compute localization loss
                l_loc += self.localization_loss(
                    pred_bbox[class_mask],
                    gt_bbox_encoded,
                    torch.ones_like(class_mask[class_mask])
                )
        
        # L1 regularization
        l_reg = sum(torch.abs(param).sum() for param in model_params)
        
        # Total loss
        total_loss = self.alpha * l_cls + self.beta * l_loc + self.gamma * l_reg
        
        loss_dict = {
            'total': total_loss.item(),
            'cls': l_cls.item(),
            'loc': l_loc if isinstance(l_loc, float) else l_loc.item(),
            'reg': l_reg.item()
        }
        
        return total_loss, loss_dict


# Example usage
if __name__ == "__main__":
    # Network parameters
    num_classes = 4  # Background, DoNotCare, Car_front, Car_side
    state_dim = 300
    T = 3
    
    # Create model and loss
    model = PointGNN(num_classes=num_classes, state_dim=state_dim, T=T)
    criterion = PointGNNLoss(alpha=0.1, beta=10.0, gamma=5e-7)
    
    # Dummy data
    N = 100  # number of vertices
    E = 500  # number of edges
    
    s_0 = torch.randn(N, state_dim)
    coords = torch.randn(N, 3)
    edge_index = torch.randint(0, N, (2, E))
    
    # Forward pass
    class_probs, bbox_preds = model(s_0, coords, edge_index)
    
    # Dummy ground truth
    gt_labels = torch.randint(0, num_classes, (N,))
    gt_bboxes = torch.randn(N, 7)
    is_object_mask = (gt_labels > 0).float()  # non-background
    
    scale_factors = {
        'l_m': 3.88, 'h_m': 1.5, 'w_m': 1.63,
        'theta_0': 0.0, 'theta_m': 1.57
    }
    
    # Compute loss
    loss, loss_dict = criterion(
        class_probs, bbox_preds, gt_labels, gt_bboxes,
        coords, is_object_mask, model.parameters(), scale_factors
    )
    
    print(f"Total Loss: {loss.item():.4f}")
    print(f"Loss breakdown: {loss_dict}")